# Fine-Tune Gemma 4 on Amazon Bestselling Books
**Goal:** Train a model that can answer questions about Amazon's top 500 bestselling books.

**GPU:** T4 x2 (free on Kaggle)

In [ ]:
!pip install unsloth kagglehub -q

In [ ]:
# Download dataset
import kagglehub
path = kagglehub.dataset_download('shambhurajejagadale/amazon-bestselling-books-dataset-500-books')
print('Dataset:', path)

In [ ]:
# Prepare training data (copy prepare_data.py logic here)
import csv, json, random
from collections import defaultdict
from pathlib import Path

csv_file = list(Path(path).glob('*.csv'))[0]
with open(csv_file) as f:
    books = list(csv.DictReader(f))
print(f'Loaded {len(books)} books')

# Create Q&A pairs
pairs = []
for book in books:
    pairs.append({'conversations': [
        {'role': 'user', 'content': f"Tell me about '{book['Title']}' by {book['Author']}"},
        {'role': 'assistant', 'content': f"'{book['Title']}' by {book['Author']} is a {book['Category']} book in {book['Sub-Genre']}. Rating: {book['Rating']}/5, Price: ${book['Price (USD)']}, Rank: #{book['Rank']}."}
    ]})
    pairs.append({'conversations': [
        {'role': 'user', 'content': f"Recommend a {book['Sub-Genre']} book"},
        {'role': 'assistant', 'content': f"I recommend '{book['Title']}' by {book['Author']} — a {book['Sub-Genre']} bestseller ranked #{book['Rank']} with {book['Rating']}/5 rating."}
    ]})

# Add top-10 list
top10 = '\n'.join(f"{b['Rank']}. {b['Title']} by {b['Author']}" for b in books[:10])
pairs.append({'conversations': [
    {'role': 'user', 'content': 'List bestseller books of Amazon'},
    {'role': 'assistant', 'content': f'Top 10 Amazon Bestsellers:\n\n{top10}'}
]})

with open('/kaggle/working/train.jsonl', 'w') as f:
    for p in pairs:
        f.write(json.dumps(p) + '\n')
print(f'Created {len(pairs)} training examples')

In [ ]:
# Load model
from unsloth import FastModel
model, tokenizer = FastModel.from_pretrained(
    'unsloth/gemma-4-E4B-it', max_seq_length=2048, load_in_4bit=True
)
model = FastModel.get_peft_model(model, r=16, lora_alpha=16)

In [ ]:
# Prepare dataset
from unsloth.chat_templates import get_chat_template, standardize_data_formats, train_on_responses_only
from datasets import load_dataset

tokenizer = get_chat_template(tokenizer, chat_template='gemma-4')
dataset = load_dataset('json', data_files='/kaggle/working/train.jsonl', split='train')
dataset = standardize_data_formats(dataset)

def fmt(examples):
    return {'text': [tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False) for c in examples['conversations']]}
dataset = dataset.map(fmt, batched=True)
print(f'Training on {len(dataset)} examples')

In [ ]:
# Train
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field='text', per_device_train_batch_size=2,
        gradient_accumulation_steps=4, warmup_steps=10, max_steps=150,
        learning_rate=2e-4, logging_steps=10, optim='adamw_8bit',
        output_dir='/kaggle/working/output', report_to='none',
    ),
)
trainer = train_on_responses_only(trainer, instruction_part='<|turn>user\n', response_part='<|turn>model\n')
trainer.train()

In [ ]:
# Test
messages = [{'role': 'user', 'content': 'List bestseller books of Amazon'}]
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors='pt').to('cuda')
outputs = model.generate(**inputs, max_new_tokens=300, temperature=0.3)
print(tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True))

In [ ]:
# Export GGUF
model.save_pretrained_gguf('/kaggle/working/gguf', tokenizer, quantization_method='q4_k_m')
print('Done! Download from Kaggle Output tab → load in Ollama locally')